# MBA Training on Kaggle

This notebook trains the MBA (Multi-objective Balanced Algorithm) recommendation model on Kaggle datasets from `hngphongkiu/rs-mba` and `hngphongkiu/otto-mba`.

## Overview
- **Code**: `/kaggle/input/datasets/hngphongkiu/rs-mba`
- **Data**: `/kaggle/input/datasets/hngphongkiu/otto-mba`
- **Output**: `/kaggle/working`

## 1. Setup and Configuration

Set up working directories and configure paths for Kaggle environment.

In [ ]:
import sys
import os

# ==================== SETUP PATHS ====================
MBA_CODE_DIR = "/kaggle/input/datasets/hngphongkiu/rs-mba"
DATA_DIR = "/kaggle/input/datasets/hngphongkiu/otto-mba"
OUTPUT_DIR = "/kaggle/working"

# File prefix: otto_pv_train.csv, otto_buy_train.csv, etc.
FILE_PREFIX = 'otto'

print("\n" + "="*70)
print("MBA Training - OTTO Dataset")
print("="*70)
print(f"File prefix: {FILE_PREFIX}")
print(f"Code: {MBA_CODE_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")

# Setup paths
sys.path.insert(0, MBA_CODE_DIR)
os.chdir(MBA_CODE_DIR)

# Verify directories
if not os.path.exists(MBA_CODE_DIR):
    raise FileNotFoundError(f"Code directory not found: {MBA_CODE_DIR}")
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

print("[OK] All directories verified\n")

## 2. Data File Configuration

Define data file names based on dataset type. Supports: otto, beibei, taobao

In [ ]:
# ==================== DATA FILE VERIFICATION & SETUP ====================

TRAIN_PV_FILE = f"{FILE_PREFIX}_pv_train.csv"
TRAIN_BUY_FILE = f"{FILE_PREFIX}_buy_train.csv"
TEST_PV_FILE = f"{FILE_PREFIX}_pv_test.csv"
TEST_BUY_FILE = f"{FILE_PREFIX}_buy_test.csv"

print(f"Verifying data files in: {DATA_DIR}")
data_files = [
    (TRAIN_PV_FILE, "Train PV"),
    (TRAIN_BUY_FILE, "Train Buy"),
    (TEST_PV_FILE, "Test PV"),
    (TEST_BUY_FILE, "Test Buy"),
]

all_exist = True
for file_name, name in data_files:
    file_path = os.path.join(DATA_DIR, file_name)
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  [OK] {name:12} - {size_mb:8.2f} MB")
    else:
        print(f"  [ERROR] {name:12} - NOT FOUND")
        all_exist = False

if not all_exist:
    raise FileNotFoundError(f"Missing data files in {DATA_DIR}")

# Create expected directory structure for MBA training code
# MBA expects: datadir/dataset/{dataset}_pv_train.csv
TRAINING_DATA_DIR = os.path.join(OUTPUT_DIR, "data", FILE_PREFIX)
os.makedirs(TRAINING_DATA_DIR, exist_ok=True)

print(f"\nSetting up training data structure:")
print(f"  Target: {TRAINING_DATA_DIR}")

# Create symlinks for data files
for file_name in [TRAIN_PV_FILE, TRAIN_BUY_FILE, TEST_PV_FILE, TEST_BUY_FILE]:
    src = os.path.join(DATA_DIR, file_name)
    dst = os.path.join(TRAINING_DATA_DIR, file_name)
    
    if not os.path.exists(dst):
        try:
            os.symlink(src, dst)
            print(f"  [OK] Linked: {file_name}")
        except Exception as e:
            print(f"  [WARNING] Could not create symlink, will copy instead: {e}")
            import shutil
            shutil.copy2(src, dst)
            print(f"  [OK] Copied: {file_name}")
    else:
        print(f"  [OK] Already exists: {file_name}")

print("[OK] All data files verified and linked\n")

## 3. Import MBA Training Libraries

Import the MBA training module and verify successful import.

In [ ]:
try:
    from main import main
    print("[OK] Successfully imported MBA training code\n")
except Exception as e:
    print(f"[ERROR] Failed to import MBA code: {e}")
    import traceback
    traceback.print_exc()
    raise

# ==================== PATCH: Fix unequal user counts ====================
import data_utils
import main as main_module
from collections import defaultdict
import pandas as pd
import numpy as np
import scipy.sparse as sp

def load_all_patched(data_path, dataset):
    """Patched version that handles unequal user counts"""
    train_pv = f"{data_path}{dataset}_pv_train.csv"
    test_pv = f"{data_path}{dataset}_pv_test.csv"
    train_buy = f"{data_path}{dataset}_buy_train.csv"
    test_buy = f"{data_path}{dataset}_buy_test.csv"
    
    user_pos_dict_buy = defaultdict(list)
    user_pos_dict_pv = defaultdict(list)
    
    # Load PV data
    train_data = pd.read_csv(train_pv, sep='\t', header=0, names=['user', 'item'],
                             usecols=[0, 1], dtype={0: np.int32, 1: np.int32})
    test_data = pd.read_csv(test_pv, sep='\t', header=0, names=['user', 'item'],
                            usecols=[0, 1], dtype={0: np.int32, 1: np.int32})
    train_aux = pd.read_csv(train_buy, sep='\t', header=0, names=['user', 'item'],
                            usecols=[0, 1], dtype={0: np.int32, 1: np.int32})
    train_data = pd.concat([train_data, test_data, train_aux], ignore_index=True)
    train_data.drop_duplicates(inplace=True, ignore_index=True)
    
    user_num = train_data['user'].max() + 1
    item_num = train_data['item'].max() + 1
    num_interaction_pv = len(train_data)
    print(f"pv: user_num:{user_num}, item_num:{item_num}, interaction:{num_interaction_pv}")
    
    train_data_pv = train_data.values.tolist()
    train_data_dict_pv = defaultdict(list)
    train_mat_pv = sp.dok_matrix((user_num, item_num), dtype=np.float32)
    for x in train_data_pv:
        uid, iid = x[0], x[1]
        train_mat_pv[uid, iid] = 1.0
        train_data_dict_pv[uid].append(iid)
        user_pos_dict_pv[uid].append(iid)
    
    # Load BUY data
    train_data = pd.read_csv(train_buy, sep='\t', header=0, names=['user', 'item'],
                             usecols=[0, 1], dtype={0: np.int32, 1: np.int32})
    num_interaction_buy = len(train_data)
    print(f"buy: user_num:{user_num}, item_num:{item_num}, interaction:{num_interaction_buy}")
    
    train_data_buy = train_data.values.tolist()
    train_data_dict_buy = defaultdict(list)
    train_mat_buy = sp.dok_matrix((user_num, item_num), dtype=np.float32)
    for x in train_data_buy:
        uid, iid = x[0], x[1]
        train_mat_buy[uid, iid] = 1.0
        train_data_dict_buy[uid].append(iid)
        user_pos_dict_buy[uid].append(iid)
    
    # Load test BUY data
    test_data = pd.read_csv(test_buy, sep='\t', header=0, names=['user', 'item'],
                            usecols=[0, 1], dtype={0: np.int32, 1: np.int32})
    print(f"number of buy test: {len(test_data)}")
    
    test_data_buy = test_data.values.tolist()
    test_data_dict_buy = defaultdict(list)
    for x in test_data_buy:
        uid, iid = x[0], x[1]
        test_data_dict_buy[uid].append(iid)
    
    # FIXED: Don't fail on unequal user counts, just warn
    pv_users = len(train_data_dict_pv)
    buy_users = len(train_data_dict_buy)
    if pv_users != buy_users:
        print(f"\n[WARNING] Unequal user counts: PV={pv_users}, BUY={buy_users}")
        print(f"[INFO] This is normal for real datasets. Proceeding...\n")
    
    return user_num, item_num, \
           train_mat_pv, train_mat_buy, \
           user_pos_dict_pv, user_pos_dict_buy, \
           train_data_dict_pv, train_data_dict_buy, \
           test_data_dict_buy

# Apply patch to both modules
data_utils.load_all = load_all_patched
main_module.load_all = load_all_patched
print("[OK] Applied patch for user count assertion\n")

## 4. Configure Training Parameters

Set up all hyperparameters for model training.

In [ ]:
# Training parameters
# Note: datadir should be parent of dataset folder
# MBA code constructs: datadir/dataset/ to find data files
TRAINING_DATADIR = os.path.join(OUTPUT_DIR, "data")

param = {
    'datadir': TRAINING_DATADIR,         # /kaggle/working/data
    'folder': OUTPUT_DIR,
    'dataset': FILE_PREFIX,              # otto (subdirectory name)
    'model': 'MF',
    'h_model': 'MF',
    'train_method': 'mba',
    'pretrain_model': 'MF',
    
    'epochs': 400,
    'batch_size': 2048,
    'C_1': 1000,
    'C_2': 1,
    'alpha': 1000,
    'lambda0': 1e-4,
    'lambda1': 1e-6,
    'beta': 0.7,
    
    'seed': 2020,
    'device': 'cuda',
    'dropout': 0.0,
    'lr': 0.001,
    'factor_num': 32,
    'num_layers': 3,
    'NSR': 1,
    'emb_dim': 32,
    'early_stop_rounds': 30,
    'pretrain_early_stop_rounds': 20,
    'top_k': [10, 20],
    'denoise_type': 'DP',
    'save_model': 1,
    'idx': 0,
    'test_only': False,
}

print("Training Configuration:")
print(f"  Data dir: {param['datadir']}")
print(f"  Dataset: {param['dataset']}")
print(f"  Model: {param['model']}")
print(f"  Epochs: {param['epochs']}")
print(f"  Batch size: {param['batch_size']}")
print(f"  Device: {param['device']}\n")
print(f"  Will load from: {param['datadir']}/{param['dataset']}/\n")

## 5. Train the Model

Run the MBA training with the configured parameters.

In [ ]:
print("\n" + "="*70)
print("Starting MBA Training")
print("="*70 + "\n")

try:
    main(param)
    print("\n" + "="*70)
    print("[OK] Training completed successfully!")
    print("="*70)
    print(f"Output saved to: {OUTPUT_DIR}\n")
except Exception as e:
    print(f"\n[ERROR] Training failed: {e}\n")
    import traceback
    traceback.print_exc()

## 6. Check Output Files

Verify that the trained models and results were saved correctly.

In [ ]:
import os
import glob

print(f"Checking output directory: {OUTPUT_DIR}\n")

# List all files in output directory
output_files = os.listdir(OUTPUT_DIR)
print(f"Files in output directory ({len(output_files)} total):")
for f in sorted(output_files):
    file_path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  - {f:<50} ({size_mb:>8.2f} MB)")
    else:
        print(f"  - {f}/ (directory)")

# Look for specific model files
print("\nModel files:")
model_files = glob.glob(os.path.join(OUTPUT_DIR, "*.pt")) + \
              glob.glob(os.path.join(OUTPUT_DIR, "*.pth"))
if model_files:
    for mf in sorted(model_files):
        print(f"  - {os.path.basename(mf)}")
else:
    print("  No model files (.pt/.pth) found")

In [ ]:
import torch
import pandas as pd
import numpy as np
from model import MF
from LightGCN import LightGCN

print("\n" + "="*70)
print("Generating Predictions")
print("="*70 + "\n")

# Load trained model
model_name = param['model']
pretrain_model = param['pretrain_model']
data_name = param['dataset']
seed = param['seed']
lambda1 = str(param['lambda1'])

model_path = os.path.join(OUTPUT_DIR,
                          f"{data_name}_target_{model_name}_{seed}_{lambda1}.pt")

print(f"Loading model from: {model_path}")

if not os.path.exists(model_path):
    print(f"[ERROR] Model file not found: {model_path}")
    print(f"Please train the model first!")
else:
    # Create model
    if model_name == 'MF':
        model = MF(train_dataset.user_num, train_dataset.item_num, param.get("factor_num", 32))
    else:
        norm_adj_buy = sp.load_npz(os.path.join(TRAINING_DATA_DIR, f'{FILE_PREFIX}_s_pre_adj_mat_buy.npz'))
        model = LightGCN(train_dataset.user_num, train_dataset.item_num,
                        norm_adj_buy, latent_dim=param['emb_dim'], n_layers=param['num_layers'],
                        device=param['device'], dropout=param['dropout'])
    
    # Load weights
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
    model.eval()
    model.to(param['device'])
    
    print("[OK] Model loaded successfully")
    
    # Generate predictions for all test users
    print("\nGenerating predictions for test users...")
    
    test_users = np.array(list(test_data_dict_buy.keys()))
    top_k = param['top_k']
    max_k = max(top_k)
    batch_size = 4096
    
    predictions = []
    
    with torch.no_grad():
        for i in range(0, len(test_users), batch_size):
            batch_users = test_users[i:i+batch_size]
            batch_users_tensor = torch.LongTensor(batch_users).to(param['device'])
            
            # Get model predictions
            if model_name == 'MF':
                # MF: user_emb @ item_emb.T
                user_embeddings = model.embed_user(batch_users_tensor)  # (batch, factor_num)
                item_embeddings = model.embed_item.weight.data  # (item_num, factor_num)
                scores = torch.matmul(user_embeddings, item_embeddings.t())  # (batch, item_num)
            else:
                # LightGCN
                scores = model.getUsersRating(batch_users_tensor)
            
            # Get top-k items
            top_scores, top_items = torch.topk(scores, k=min(max_k, scores.shape[1]))
            
            for user_idx, user_id in enumerate(batch_users):
                for rank, item_id in enumerate(top_items[user_idx].cpu().numpy(), 1):
                    score = top_scores[user_idx, rank-1].cpu().item()
                    predictions.append({
                        'user_id': user_id,
                        'item_id': int(item_id),
                        'rank': rank,
                        'score': score
                    })
            
            print(f"  Processed {min(i+batch_size, len(test_users))}/{len(test_users)} users")
    
    # Save predictions to CSV
    pred_df = pd.DataFrame(predictions)
    pred_file = os.path.join(OUTPUT_DIR, f"predictions_{data_name}_{seed}.csv")
    pred_df.to_csv(pred_file, index=False)
    
    print(f"\n[OK] Predictions saved to: {pred_file}")
    print(f"Total predictions: {len(predictions)}")
    print(f"Sample predictions:")
    print(pred_df.head(10))


## 7. Generate Predictions

Generate prediction file (CSV) for test data using trained model.
